<a href="https://colab.research.google.com/github/nedokormysh/Stepik_LLM/blob/week_3_langchain/3_2_%D0%A0%D0%B5%D1%88%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D1%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain openai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.1/809.1 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.5/257.5 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.9/260.9 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.5/138.5 kB 891.5 kB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
from tqdm import tqdm
from getpass import getpass

from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, TransformChain
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

## Если используете ключ от OpenAI, запустите эту ячейку 👇

In [ ]:
import os
from langchain.chat_models import ChatOpenAI


# os.environ['OPENAI_API_KEY'] = "Введите ваш OpenAI API ключ"
os.environ['OPENAI_API_KEY'] = getpass(prompt='Введите ваш OpenAI API ключ')

# Инициализируем языковую модель
llm = ChatOpenAI(temperature=0.0)

## Если используете ключ из курса, запустите эти ячейки 👇


In [ ]:
!wget https://raw.githubusercontent.com/a-milenkin/LLM_practical_course/main/notebooks/utils.py

--2024-03-19 08:40:53--  https://raw.githubusercontent.com/a-milenkin/LLM_practical_course/main/notebooks/utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10822 (11K) [text/plain]
Saving to: ‘utils.py’

utils.py            100%[===================>]  10.57K  --.-KB/s    in 0s      

2024-03-19 08:40:53 (78.3 MB/s) - ‘utils.py’ saved [10822/10822]



In [ ]:
from utils import ChatOpenAI
from getpass import getpass

#course_api_key= "Введите ваш API ключ, полученный в боте курса"
course_api_key = getpass(prompt='Введите ваш API ключ, полученный в боте курса')

# инициализируем языковую модель
llm = ChatOpenAI(temperature=0.0, course_api_key=course_api_key)

Введите ваш API ключ, полученный в боте курса··········


## Задание 3.2.9 🤔 Кажется, это что-то на LLM-ском? 🧐

In [ ]:
df = pd.read_csv("https://stepik.org/media/attachments/lesson/1110883/raw_texts.csv")
df.head()

,raw_text
0,"The sun was setting, casting long shadows over..."
1,"Le soleil se couchait, jetant de longues ombre..."
2,"El sol se estaba poniendo, proyectando largas ..."
3,"La ciudad estaba llena de vida, sus calles lle..."
4,"La ville était pleine de vie, ses rues remplie..."


Напишем функцию, которая очистит текст от ненужных символов: `¿, ¡, £`

In [ ]:
def clean_text(inputs: dict) -> dict:
    text = inputs["text"]

    text = re.sub('[¿¡£]', '', text)

    return {"output_text": text}

Будем просить у модели определять язык и имя главного персонажа и выдавать ответ в виде словаря. Для этого создадим `Output parser`, с которым вы уже познакомились в прошлых уроках.

## test

In [ ]:
strng = 'The sun was setting, casting long shadows over the small town. Jo¿hn, a middle-aged man with a heart full of dreams, was sitting on the porch of his old house. His friends'

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

In [ ]:
# создадим шаблон и промпт
language_template = "What language is the following text written in?\n{output_text}"
language_prompt = PromptTemplate(template=language_template, input_variables=["output_text"])

In [ ]:
# создаём цепочку
language_chain = LLMChain(llm=llm, prompt=language_prompt, output_key='final_output')

In [ ]:
# создаём преобразующую цепочку
from langchain.chains import TransformChain

text_clean_chain = TransformChain(input_variables=["text"],
                                  output_variables=["output_text"],
                                  transform=clean_text)

In [ ]:
text_clean_chain.run(strng)

'The sun was setting, casting long shadows over the small town. John, a middle-aged man with a heart full of dreams, was sitting on the porch of his old house. His friends'

In [ ]:
from langchain.chains import SequentialChain

In [ ]:
sequential_chain = SequentialChain(chains=[text_clean_chain, language_chain],
                                   input_variables=['text'],
                                   output_variables=['final_output'])

In [ ]:
sequential_chain.run({'text': strng})

'English'

### dirty work (решение)

#### language

In [ ]:
language = []
for l in df['raw_text']:
  language.append(sequential_chain.run({'text': l}))

In [ ]:
language

['English',
 'French',
 'Spanish',
 'Spanish',
 'French',
 'German',
 'German',
 'Russian',
 'English',
 'Spanish',
 'French',
 'Russian',
 'Italian']

In [ ]:
# ['English',
#  'French',
#  'Spanish',
#  'Spanish',
#  'French',
#  'German',
#  'German',
#  'Russian',
#  'English',
#  'Spanish',
#  'French',
#  'Russian',
#  'Italian']

In [ ]:
df_result=pd.DataFrame()

In [ ]:
df_result['language'] = language

#### clean_text

In [ ]:
clean_text = []
for l in df['raw_text']:
  clean_text.append(text_clean_chain.run({'text': l}))

In [ ]:
df_result['text'] = clean_text

In [ ]:
df_result = df_result[['text', 'language']]

In [ ]:
df_result

,text,language
0,"The sun was setting, casting long shadows over...",English
1,"Le soleil se couchait, jetant de longues ombre...",French
2,"El sol se estaba poniendo, proyectando largas ...",Spanish
3,"La ciudad estaba llena de vida, sus calles lle...",Spanish
4,"La ville était pleine de vie, ses rues remplie...",French
5,"Die Stadt war voller Leben, ihre Straßen gefül...",German
6,Die Sonne ging unter und warf lange Schatten ü...,German
7,"В тихом уголке старого города, где узкие улочк...",Russian
8,In a small town nestled between the mountains ...,English
9,En un pequeño pueblo situado entre las montaña...,Spanish


#### main character

In [ ]:
# Определяем шаблон для определения имени главного героя
main_character_template = "Who is the main character in the following text? Return only name in original language of text\n{output_text}"
main_character_prompt = PromptTemplate(template=main_character_template, input_variables=["output_text"])

In [ ]:
# создаём цепочку
character_chain = LLMChain(llm=llm, prompt=main_character_prompt, output_key='final_output')

In [ ]:
sequential_chain = SequentialChain(chains=[text_clean_chain, character_chain],
                                   input_variables=['text'],
                                   output_variables=['final_output'])

In [ ]:
sequential_chain.run({'text': strng})

'John'

In [ ]:
character = []
for l in df['raw_text']:
  character.append(sequential_chain.run({'text': l}))

In [ ]:
df_result['main_character'] = character

In [ ]:
df_result

,text,language,main_character
0,"The sun was setting, casting long shadows over...",English,John
1,"Le soleil se couchait, jetant de longues ombre...",French,Pierre
2,"El sol se estaba poniendo, proyectando largas ...",Spanish,Carlos
3,"La ciudad estaba llena de vida, sus calles lle...",Spanish,Juan
4,"La ville était pleine de vie, ses rues remplie...",French,Jean
5,"Die Stadt war voller Leben, ihre Straßen gefül...",German,Johann
6,Die Sonne ging unter und warf lange Schatten ü...,German,Hans
7,"В тихом уголке старого города, где узкие улочк...",Russian,Анна
8,In a small town nestled between the mountains ...,English,Laura
9,En un pequeño pueblo situado entre las montaña...,Spanish,Maria


In [ ]:
df_result[['text', 'language', 'main_character']].to_csv('3.2.9_solution.csv', index=False)

### LCEL

In [ ]:
# Немного перепишем функцию очистки
def clean_text(inputs: dict) -> dict:
    text = inputs["output_text"]

    # text = inputs["text"]

    text = re.sub('[¿¡£]', '', text)

    return {"output_text": text}

In [ ]:
seq_chain = clean_text | language_prompt | llm # И больше ничего не нужно!!!

seq_chain.invoke({'output_text': strng})

AIMessage(content='English', response_metadata={'finish_reason': 'stop', 'logprobs': None})

In [ ]:
from langchain.schema.output_parser import StrOutputParser

chain_with_parser = clean_text | language_prompt | llm | StrOutputParser() # И ВСЁ!

print(chain_with_parser.invoke({'output_text': strng}))

English


# Prime

In [ ]:
# Определим схемы ответа
language_schema = ...

person_schema = ...

response_schemas = [language_schema, person_schema]
output_parser = ... # Создаём парсер и подаём в него список со схемами
format_instructions = ... # Получаем инструкции по форматированию ответа

Напишем шаблон промпта со своим вопросом и инструкциями по форматированию ответа. Будем передавать в этот промпт сырой текст

In [ ]:
prompt_template = "Ваша задача - определить язык и имя главного персонажа в следующем тексте.\n\n{text}\n\nОтветьте в формате: language: {language} main_character: {main_character} на английском языке."

# prompt = PromptTemplate(
#     #YOUR CODE HERE
# )

prompt = PromptTemplate(
  template=prompt_template,
  input_variables=["text"],
)

Создадим цепочку с помощью `LCEL`

In [ ]:
chain = clean_text | prompt | llm

In [ ]:
for text in tqdm(df['raw_text']):
    # YOUR CODE HERE
    answer = chain.invoke({'output_text': text, 'style': style})
    break # Для отладки. Уберите, когда убедитесь, что на одном примере работает

Сохраним всё в итоговый файл. Убедитесь, что на этом этапе у вас в столбцах

- `text` - очищенный текст (без символов ¿, ¡, £)
- `language` - язык, на котором написан текст (название языка указать на английском языке)
- `main_character` - имя главного персонажа в тексте (указать на том языке, на котором и написан сам текст)

In [ ]:
df[['text', 'language', 'main_character']].to_csv('3.2.9_solution.csv', index=False)

## Mistral

Создадим функцию для очистки текста от странных символов:

In [ ]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'[¿¡£]', '', text)
    return text

Создадим цепочку для очистки текста:

In [ ]:
from langchain.chains import TransformChain

# clean_text_chain = TransformChain(
#     input_variables=["text"],
#     output_variables=["cleaned_text"],
#     transform=clean_text
# )

def clean_text_chain(inputs: dict) -> dict:
    text = inputs["text"]
    cleaned_text = clean_text(text)
    return {"cleaned_text": cleaned_text}

Создадим цепочку для определения языка текста:

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

language_prompt = PromptTemplate(
    input_variables=["cleaned_text"],
    template="What is the language of the following text: {cleaned_text}"
)

language_chain = LLMChain(llm=llm, prompt=language_prompt, output_key="language")

Создадим цепочку для определения имени главного героя:

In [ ]:
main_character_prompt = PromptTemplate(
    input_variables=["cleaned_text"],
    template="Who is the main character in the following text: {cleaned_text}"
)

main_character_chain = LLMChain(llm=llm, prompt=main_character_prompt, output_key="main_character")

Создадим последовательную цепочку для выполнения всех задач:

In [ ]:
from langchain.chains import SequentialChain

seq_chain = SequentialChain(
    chains=[clean_text_chain, language_chain, main_character_chain],
    input_variables=["text"],
    output_variables=["cleaned_text", "language", "main_character"]
)

Применим цепочку к данным:

In [ ]:
# import pandas as pd

# data = []
# for i, row in df:
#     result = seq_chain.invoke({"text": row["raw_text"]})
#     data.append(result)
#     break

# df_result = pd.DataFrame(data)
# df_result.to_csv("cleaned_texts.csv", index=False)

ValueError: too many values to unpack (expected 2)

In [ ]:
data = []
for text in tqdm(df['raw_text']):
  result = seq_chain.invoke({"text": text})
  data.append(result)
  break

  0%|          | 0/13 [00:00<?, ?it/s]


TypeError: expected string or bytes-like object

In [ ]:
for text in df['raw_text']:
    print(type(text))
    break

<class 'str'>


In [ ]:
for text in tqdm(df['raw_text']):
    print(type(text))
    break

  0%|          | 0/13 [00:00<?, ?it/s]

<class 'str'>


In [ ]:
for i, row in df.iterrows():
  print(row)
  break

raw_text    The sun was setting, casting long shadows over...
Name: 0, dtype: object


In [ ]:
strng = 'The sun was setting, casting long shadows over John'
seq_chain.invoke({"text": strng})

TypeError: expected string or bytes-like object

In [ ]:
text = 'In the German novel "Die Leiden des jungen Werthers" by Johann Wolfgang von Goethe, the main character is Werther.'
result = seq_chain.invoke({"text": text})
print(result)

TypeError: expected string or bytes-like object

In [ ]:
# import re
# import pandas as pd
# from langchain.chains import SequentialChain, LLMChain
# from langchain.prompts import PromptTemplate
# from langchain.schema.output_parser import StrOutputParser

# # Определяем функцию для очистки текста
# def clean_text(text):
#     text = re.sub(r'[¿¡£]', '', text)
#     return text

# # Определяем шаблон для определения языка
# language_template = '''В каком языке написан следующий текст: {cleaned_text}?
# Ответ в формате: Language: <language>'''
# language_prompt = PromptTemplate(input_variables=['cleaned_text'], template=language_template)

# # Определяем шаблон для определения имени главного героя
# main_character_template = '''Кто главный герой в следующем тексте: {cleaned_text}?
# Ответ в формате: Main character: <main_character>'''
# main_character_prompt = PromptTemplate(input_variables=['cleaned_text'], template=main_character_template)

# # Создаем цепочку для очистки текста
# clean_text_chain = LLMChain(llm=llm, prompt=PromptTemplate(input_variables=['text'], template='{text}'), output_key='cleaned_text', transform=clean_text)

# # Создаем цепочку для определения языка
# language_chain = LLMChain(llm=llm, prompt=language_prompt, output_key='language', output_parser=StrOutputParser())

# # Создаем цепочку для определения имени главного героя
# main_character_chain = LLMChain(llm=llm, prompt=main_character_prompt, output_key='main_character', output_parser=StrOutputParser())

# # Создаем последовательную цепочку
# seq_chain = SequentialChain(
#     chains=[clean_text_chain, language_chain, main_character_chain],
#     input_variables=["text"],
#     output_variables=["cleaned_text", "language", "main_character"]
# )

# # Считываем данные
# df = pd.read_csv("https://stepik.org/media/attachments/lesson/1110883/raw_texts.csv")

# # Применяем цепочку к каждому тексту в датасете
# results = []
# for i, row in df.iterrows():
#     result = seq_chain.invoke({"text": row["text"]})
#     results.append(result)
#     break

# # Сохраняем результаты в csv-файл
# df_results = pd.DataFrame(results)
# df_results.to_csv("results.csv", index=False)


## Test Mistral

In [ ]:
import pandas as pd
import re
from langchain.chains import SequentialChain, LLMChain
from langchain.prompts import PromptTemplate

# Считываем данные
df = pd.read_csv("https://stepik.org/media/attachments/lesson/1110883/raw_texts.csv")

In [ ]:
df.shape

(13, 1)

In [ ]:
def clean_text(inputs: dict) -> dict:
    text = inputs["text"]
    text = re.sub('[¿¡£]', '', text)
    return {"output_text": text}

# Определяем шаблон для определения языка
language_template = "What language is the following text written in?\n{output_text}"
language_prompt = PromptTemplate(template=language_template, input_variables=["output_text"])

# Определяем шаблон для определения имени главного героя
main_character_template = "Who is the main character in the following text?\n{output_text}"
main_character_prompt = PromptTemplate(template=main_character_template, input_variables=["output_text"])

# Создаём цепочку для очистки текста

text_clean_chain = TransformChain(input_variables=["text"],
                                  output_variables=["output_text"],
                                  transform=clean_text)

# Создаём цепочку для определения языка
language_chain = LLMChain(llm=llm, prompt=language_prompt)

# Создаём цепочку для определения имени главного героя
main_character_chain = LLMChain(llm=llm, prompt=main_character_prompt)

# Создаём последовательную цепочку
seq_chain = SequentialChain(
    chains=[text_clean_chain, language_chain, main_character_chain],
    input_variables=["text"],
    output_variables=["language", "main_character"]
)


# Применяем цепочку к каждому тексту в датафрейме
df[["language", "main_character"]] = df["text"].apply(lambda text: seq_chain.invoke(text))

# Сохраняем результаты в csv-файл
df.to_csv("cleaned_texts.csv", index=False)

ValidationError: 1 validation error for SequentialChain
__root__
  Chain returned keys that already exist: {'text'} (type=value_error)

In [ ]:
import pandas as pd
import re
from langchain.chains import SequentialChain, LLMChain
from langchain.prompts import PromptTemplate

# Считываем данные
df = pd.read_csv("https://stepik.org/media/attachments/lesson/1110883/raw_texts.csv")

# Определяем функцию для очистки текста
# def clean_text(text):
#     text = re.sub(r'[¿¡£]', '', text)
#     return text

def clean_text(inputs: dict) -> dict:
    text = inputs["text"]
    text = re.sub('[¿¡£]', '', text)
    return {"output_text": text}

# Определяем шаблон для определения языка
language_template = "What language is the following text written in?\n{text}"
language_prompt = PromptTemplate(template=language_template, input_variables=["output_text"])

# Определяем шаблон для определения имени главного героя
main_character_template = "Who is the main character in the following text?\n{text}"
main_character_prompt = PromptTemplate(template=main_character_template, input_variables=["output_text"])

# Создаём цепочку для очистки текста
# clean_text_chain = LLMChain(llm=llm, prompt=PromptTemplate(template="{text}", input_variables=["text"]), transform=clean_text)
text_clean_chain = TransformChain(input_variables=["text"],
                                  output_variables=["output_text"],
                                  transform=clean_text)

# Создаём цепочку для определения языка
language_chain = LLMChain(llm=llm, prompt=language_prompt)

# Создаём цепочку для определения имени главного героя
main_character_chain = LLMChain(llm=llm, prompt=main_character_prompt)

# Создаём последовательную цепочку
seq_chain = SequentialChain(
    chains=[text_clean_chain, language_chain, main_character_chain],
    input_variables=["text"],
    output_variables=["language", "main_character"]
)


# Применяем цепочку к каждому тексту в датафрейме
df[["language", "main_character"]] = df["text"].apply(lambda text: seq_chain.invoke(text))

# Сохраняем результаты в csv-файл
df.to_csv("cleaned_texts.csv", index=False)

ValidationError: 1 validation error for SequentialChain
__root__
  Chain returned keys that already exist: {'text'} (type=value_error)

In [ ]:
seq_chain = SequentialChain(
    chains=[
        LLMChain(llm=llm, prompt=PromptTemplate(template="{text}", input_variables=["text"])),
        LLMChain(llm=llm, prompt=language_prompt),
        LLMChain(llm=llm, prompt=main_character_prompt)
    ],
    input_variables=["text"],
    output_variables=["language", "main_character"]
)

text = 'In the German novel "Die Leiden des jungen Werthers" by Johann Wolfgang von Goethe, the main character is Werther.'
result = seq_chain.invoke({"text": text})
print(result)

ValidationError: 1 validation error for SequentialChain
__root__
  Chain returned keys that already exist: {'text'} (type=value_error)